# Voice-over → word times (Whisper), for any MiC video

Finds the time of every word in each beat recording, so `on "word"` lines in the MiC Scene Editor (and the video's cues) land on your real voice.

**Set-up:**
1. Name each recording after its beat: `S1-B1.wav`, `S2-B10.m4a`, `s4_b3.mp3`, `S01-B02.mp3` all work (read as S1-B1, S2-B10, S4-B3, S1-B2). WAV, MP3, M4A, FLAC, OGG, AAC, OPUS and WEBM are fine.
2. Upload them as a **private Kaggle dataset**. Optional but useful: put your `.mic` scene files in the same dataset, so the notebook knows each beat's words and can retry any recording it heard badly.
3. In this notebook: **Add Input** → your dataset. **Settings → Internet: On**. Accelerator: **GPU T4** if available.
4. **Save Version → Save & Run All.**

**Output tab:** `voice_timings.json` (load it in the editor: Files → 🕑 Load voice_timings.json) and `word_check.csv` (each beat's length and anything worth a look).

To redo a few beats only: set `ONLY = ['S2-B3', ...]`, add your previous `voice_timings.json` to the inputs, and run. The new beats are merged into it.


In [ ]:
# Settings
MODEL = 'medium.en'   # 'small.en' is faster on CPU; 'medium.en' is more accurate (recommended on GPU)
AUDIO_DIR = ''        # leave '' to search everything under /kaggle/input
ONLY = []             # [] = every recording; or ['S2-B3', 'S4-B1'] to redo just those (merged into an earlier voice_timings.json in the inputs)
# Technical words Whisper should expect. Add this video's names and terms. (Never the script itself: given the exact
# sentences, Whisper may skip them.)
VOCAB = ('Na+/K+-ATPase, action potential, depolarisation, repolarisation, hyperpolarisation, overshoot, afterhyperpolarisation, '
         'voltage-gated, inactivation, tetrodotoxin, squid giant axon, Julius Bernstein, Alan Hodgkin, Andrew Huxley, Bernard Katz, '
         'Kenneth Cole, Walther Nernst, Goldman-Hodgkin-Katz, equilibrium potential, millivolts, millimolar, millisecond, '
         'permeability, conductance, sodium, potassium, chloride, calcium.')


In [ ]:
# faster-whisper, plus the CUDA 12 libraries it needs on the GPU (Kaggle's image may not ship cuBLAS 12 / cuDNN 9)
!pip install -q faster-whisper nvidia-cublas-cu12 'nvidia-cudnn-cu12==9.*'


In [ ]:
# Each beat's words, from the .mic files in the inputs (optional): used only to check how much of a recording was heard.
import os, re, glob, json
SCRIPT = {}
pat = re.compile(r'[sS]\s*(\d+)\s*[-_ .]?\s*[bB]\s*(\d+)')
def beat_id(name):
    m = pat.search(name)
    return f'S{int(m.group(1))}-B{int(m.group(2))}' if m else None
for f in glob.glob('/kaggle/input/**/*.mic', recursive=True):
    for line in open(f, encoding='utf-8'):
        m = re.match(r'\s*beat\s+(\S+)\s+["“](.*?)["”]', line)
        if m and beat_id(m.group(1)): SCRIPT[beat_id(m.group(1))] = m.group(2)
print(f'{len(SCRIPT)} beats of script found in .mic files' if SCRIPT else 'No .mic files in the inputs: fine, every recording is still transcribed.')


In [ ]:
AUDIO_EXT = ('.wav', '.mp3', '.m4a', '.flac', '.ogg', '.aac', '.opus', '.webm')
root = AUDIO_DIR or '/kaggle/input'
files = [p for p in glob.glob(os.path.join(root, '**', '*'), recursive=True) if p.lower().endswith(AUDIO_EXT)]
by_beat, unmatched = {}, []
for p in sorted(files):
    bid = beat_id(os.path.basename(p))
    if not bid: unmatched.append(p); continue
    if bid in by_beat: print('WARNING: two files for', bid, '->', by_beat[bid], 'and', p, '(using the first)')
    else: by_beat[bid] = p
print(f'{len(files)} audio files found, {len(by_beat)} named after a beat.')
if unmatched: print('Files with no beat ID in the name (skipped):', *unmatched, sep='\n  ')
if SCRIPT:
    missing = [b for b in SCRIPT if b not in by_beat]
    extra = [b for b in by_beat if b not in SCRIPT]
    if missing: print('Beats in the scenes with no recording:', ', '.join(missing))
    if extra: print('Recordings for beats not in the scenes:', ', '.join(extra))
if ONLY: print('Only these beats:', ', '.join(b for b in ONLY if b in by_beat) or '(none of them have a recording!)')


In [ ]:
import ctypes, glob, os, site, numpy as np
# Load the pip-installed CUDA libraries into this process so ctranslate2 can find them
# (setting LD_LIBRARY_PATH inside a running notebook has no effect).
for sp in site.getsitepackages():
    for pat in ('nvidia/cublas/lib/libcublasLt.so.12', 'nvidia/cublas/lib/libcublas.so.12', 'nvidia/cudnn/lib/libcudnn*.so.9'):
        for lib in sorted(glob.glob(os.path.join(sp, pat))):
            try: ctypes.CDLL(lib, mode=ctypes.RTLD_GLOBAL)
            except OSError as e: print('could not load', os.path.basename(lib), e)

from faster_whisper import WhisperModel
model, DEVICE = None, 'CPU'
try:
    model = WhisperModel(MODEL, device='cuda', compute_type='float16')
    # a real test: the GPU libraries are only loaded on the first transcription
    list(model.transcribe(np.zeros(16000, np.float32), language='en')[0])
    DEVICE = 'GPU'
except Exception as e:
    print('GPU not usable, falling back to CPU:', str(e)[:160])
    model = WhisperModel(MODEL, device='cpu', compute_type='int8', cpu_threads=os.cpu_count())
print('Model', MODEL, 'on', DEVICE, '(on CPU, medium.en takes roughly 1-2x the audio length; small.en is about 3x faster)')


In [ ]:
import subprocess, numpy as np, csv
SR = 16000
def load_audio(path):
    # decode with ffmpeg to 16 kHz mono float32 (Kaggle's PyAV version clashes with faster-whisper's own decoder)
    raw = subprocess.run(['ffmpeg', '-nostdin', '-v', 'error', '-i', path, '-f', 'f32le', '-ac', '1', '-ar', str(SR), '-'],
                         capture_output=True, check=True).stdout
    return np.frombuffer(raw, np.float32).copy()

def key(b):  # S1-B2 before S1-B10
    m = re.match(r'S(\d+)-B(\d+)', b); return (int(m.group(1)), int(m.group(2)))

out = {'model': MODEL, 'beats': {}}
if ONLY:  # merge into an earlier run
    prev = glob.glob('/kaggle/input/**/voice_timings.json', recursive=True)
    if prev: out['beats'] = json.load(open(prev[0]))['beats']; print('merging into', prev[0])
    else: print('No earlier voice_timings.json in the inputs: the output will hold only', ONLY)
rows = []
for bid in sorted(by_beat, key=key):
    if ONLY and bid not in ONLY: continue
    audio = load_audio(by_beat[bid])
    n_script = len(re.sub(r'\[[^\]]*\]', ' ', SCRIPT[bid]).split()) if bid in SCRIPT else None
    words, coverage = [], -1
    # plain settings first; if much of the script is missing, retry with voice detection, then without the glossary
    for attempt, kw in enumerate([dict(initial_prompt=VOCAB, vad_filter=False), dict(initial_prompt=VOCAB, vad_filter=True),
                                  dict(initial_prompt=None, vad_filter=True)]):
        segs, info = model.transcribe(audio, language='en', word_timestamps=True, beam_size=5,
                                      condition_on_previous_text=False, **kw)
        ws = [{'w': w.word.strip(), 's': round(w.start, 3), 'e': round(w.end, 3)} for s in segs for w in (s.words or [])]
        cov = min(1.0, len(ws) / n_script) if n_script else (1.0 if ws else 0.0)
        if cov > coverage: words, coverage = ws, cov
        if coverage >= 0.9: break
        print(f'  {bid}: heard {cov:.0%} on attempt {attempt + 1}, retrying')
    note = '' if coverage >= 0.9 else (f'only {coverage:.0%} of the script heard' if n_script else 'no words heard')
    out['beats'][bid] = {'file': os.path.basename(by_beat[bid]), 'duration': round(len(audio) / SR, 3),
                         'speech_start': words[0]['s'] if words else None, 'speech_end': words[-1]['e'] if words else None,
                         'cues': {}, 'transcript': ' '.join(w['w'] for w in words),
                         'words': [[w['w'], w['s'], w['e']] for w in words]}
    rows.append((bid, os.path.basename(by_beat[bid]), round(len(audio) / SR, 2), len(words), note))
    print(f'{bid:8s} {len(audio) / SR:6.2f}s  {len(words):4d} words  {note}')

json.dump(out, open('/kaggle/working/voice_timings.json', 'w'), ensure_ascii=False, indent=1)
with open('/kaggle/working/word_check.csv', 'w', newline='') as f:
    w = csv.writer(f); w.writerow(['beat', 'file', 'seconds', 'words heard', 'notes']); w.writerows(rows)
total = sum(r[2] for r in rows)
print(f'\nDone: {len(rows)} beats transcribed ({total/60:.1f} min), {len(out["beats"])} in voice_timings.json. '
      'Load it in the editor (Files → 🕑 Load voice_timings.json).')
